# Precomputing Visualisations

#### INPUT  
- narrative_extractions_normalized.csv
- preprocessed_dataset.tsv
- cluster_assignments_macro.csv / cluster_assignments_meso.csv / cluster_assignments_micro.csv
- cluster_labels_macro.csv / cluster_labels_meso.csv / cluster_labels_micro.csv
- embeddings.npy / embedding_ids.npy

#### WHAT THIS NOTEBOOK DOES
- loads the cluster assignments, labels and extractions produced by notebook 06_label.ipynb
- joins assignments, labels, metrics and embeddings from the earlier pipeline stages
- computes representative examples, averaged metrics and time buckets per cluster
- builds the entity search index, vocabulary and (e5-small) entity embeddings
- builds thematic-search artifacts: per-cluster centroids and slim per-article narrative embeddings
- builds a 2D UMAP semantic map of articles and cluster centroids for the overview
- writes the parquet / npy / json tables consumed by the Streamlit app

#### OUTPUTS 
- article_table.parquet --> one row per article (assignments + metrics + iso_week)
- cluster_meta.parquet --> one row per cluster per level (labels, sizes, avg metrics, top entities, examples)
- cluster_publisher.parquet --> one row per cluster x publisher per level
- cluster_week.parquet --> one row per cluster x iso_week per level
- slot_fillers.parquet --> per-filler rows for the six card slots
- entity_ftype.parquet --> label to dominant filler_type
- entity_index.parquet / entity_vocab.parquet / entity_embeddings.npy --> entity search index, vocabulary and embeddings
- entity_article_bridge.parquet --> (label, id) pairs for entity scoping
- search_meta.json --> records the embedding model used for search
- cluster_centroids.npy / cluster_centroid_keys.parquet --> thematic-search centroids per cluster
- narrative_emb_small.npy / narrative_emb_ids.npy --> slim per-article narrative embeddings for thematic search
- map_xy.parquet / map_centroids.parquet --> 2D semantic map coordinates (articles + centroids)


In [ ]:
import os
import numpy as np
import pandas as pd

import config.precompute_config as cfg
from lib.precompute_lib import (
    load_preprocessed, load_assignments, load_labels,
    add_transformed_metrics, add_iso_week,
    build_article_table, build_cluster_meta,
    build_cluster_publisher, build_cluster_week,
    build_slot_fillers, build_entity_ftype,
)

os.makedirs(cfg.VIZ_DIR, exist_ok=True)

In [2]:
# load inputs
preprocessed = load_preprocessed(cfg.PREPROCESSED_TSV)
assignments  = load_assignments(cfg.MACRO_ASSIGNMENTS_CSV, cfg.MESO_ASSIGNMENTS_CSV, cfg.MICRO_ASSIGNMENTS_CSV)
labels       = load_labels(cfg.MACRO_LABELS_CSV, cfg.MESO_LABELS_CSV, cfg.MICRO_LABELS_CSV)
extractions  = pd.read_csv(cfg.EXTRACTIONS_CSV)
extractions["id"] = extractions["id"].astype(str)

embeddings    = np.load(cfg.EMBEDDINGS_NPY)
embedding_ids = np.load(cfg.EMBEDDING_IDS, allow_pickle=True).astype(str)

print(f"Preprocessed: {len(preprocessed):,} articles")
print(f"Assignments:  {len(assignments):,} rows")
print(f"Labels:       macro={len(labels['macro'])}, meso={len(labels['meso'])}, micro={len(labels['micro'])}")
print(f"Embeddings:   {embeddings.shape}")

Preprocessed: 64,504 articles
Assignments:  44,876 rows
Labels:       macro=17, meso=226, micro=351
Embeddings:   (44876, 1024)


In [3]:
# transform metrics (invert flesch -> complexity) and bucket time into iso weeks
preprocessed = add_transformed_metrics(preprocessed)
preprocessed = add_iso_week(preprocessed, trim_partial=cfg.TRIM_PARTIAL_WEEKS)

print(f"After week trim: {len(preprocessed):,} articles")
print(f"Week span: {preprocessed['iso_week'].min().date()} -> {preprocessed['iso_week'].max().date()} "
      f"({preprocessed['iso_week'].nunique()} weeks)")
print(f"complexity range: {preprocessed['complexity'].min():.1f} to {preprocessed['complexity'].max():.1f}")

After week trim: 64,375 articles
Week span: 2025-06-02 -> 2026-05-25 (52 weeks)
complexity range: 20.1 to 115.9


In [ ]:
# TABLE 1: article_table
article_table = build_article_table(assignments, preprocessed)
article_table.to_parquet(cfg.ARTICLE_TABLE, index=False)
print(f"article_table: {len(article_table):,} rows -> {cfg.ARTICLE_TABLE}")
article_table.head()

article_table: 44,786 rows -> data/viz/article_table.parquet


,id,macro_cluster,meso_cluster,micro_cluster,pubtime,publisher,head,sentiment_score,linguistic_complexity,emotion_anger,emotion_fear,emotion_joy,emotion_sadness,complexity,sentiment,anger,fear,joy,sadness,iso_week
0,58672106,1.0,1008.0,1008000.0,2025-10-28 21:27:22+01,TX Group,Lawrow: «Russland könnte EU- und Nato-Sicherhe...,-0.034477,27.612069,0.568838,0.101275,0.129029,0.200859,72.387931,-0.034477,0.568838,0.101275,0.129029,0.200859,2025-10-27
1,58312844,13.0,NaN,NaN,2025-09-22 00:00:00+02,CH Media,Das entscheidet den Abstimmungskrimi des Jahres,-0.023208,56.445778,0.557288,0.073606,0.171580,0.197526,43.554222,-0.023208,0.557288,0.073606,0.171580,0.197526,2025-09-15
2,58796591,16.0,16004.0,16004001.0,2025-11-10 15:28:21+01,TX Group,Forschung und Bildung: Parmelin unterschreibt ...,-0.000537,33.232353,0.182771,0.042330,0.682311,0.092588,66.767647,-0.000537,0.182771,0.042330,0.682311,0.092588,2025-11-10
3,60345472,3.0,3024.0,3024001.0,2026-03-24 07:42:03+01,Ringier,"Einfamilienhaus evakuiert, Strassen gesperrt: ...",-0.059775,35.219968,0.240757,0.358307,0.111309,0.289627,64.780032,-0.059775,0.240757,0.358307,0.111309,0.289627,2026-03-23
4,57504722,3.0,3005.0,NaN,2025-07-10 08:15:57+02,Ringier,Transporter zurückgelassen: Velo-Diebe flüchte...,-0.029948,58.803886,0.652494,0.162049,0.039109,0.146348,41.196114,-0.029948,0.652494,0.162049,0.039109,0.146348,2025-07-07


In [5]:
# TABLE 2: cluster_meta (includes representative example selection via embeddings)
cluster_meta = build_cluster_meta(article_table, labels, extractions, embeddings, embedding_ids)
cluster_meta.to_parquet(cfg.CLUSTER_META, index=False)
print(f"cluster_meta: {len(cluster_meta):,} rows -> {cfg.CLUSTER_META}")
print(cluster_meta.groupby('level').size())
cluster_meta.head()

cluster_meta: 594 rows -> data/viz/cluster_meta.parquet
level
macro     17
meso     226
micro    351
dtype: int64


,level,cluster,parent,n_articles,corpus_share,title,description,complexity,sentiment,anger,fear,joy,sadness,primary_domain,top_entities,slot_type_groups,representative_ids
0,macro,0.0,NaN,4076,0.091011,Geopolitical Diplomacy,State and non-state actors clash over terms fo...,54.952665,-0.112224,0.536976,0.075257,0.192142,0.195626,international_diplomatic,subject::Donald Trump ¦ United States (Trump a...,"{""subject"": [{""filler_type"": ""actor"", ""categor...","57861626,57867041,59154948,57871403,59008758"
1,macro,1.0,NaN,5830,0.130175,Military Coercion,Armed actors deploy force to impose strategic ...,55.185124,-0.113533,0.546307,0.102416,0.165786,0.185491,international_security,subject::United States (Trump administration) ...,"{""subject"": [{""filler_type"": ""actor"", ""categor...","60026928,60111611,60148057,60221331,57684578"
2,macro,2.0,NaN,1151,0.025700,Identity Assertion,Communities and individuals claiming cultural ...,47.043122,-0.132520,0.439803,0.044576,0.290225,0.225396,identity_culture,subject::Donald Trump ¦ Pro-Palestinian demons...,"{""subject"": [{""filler_type"": ""actor"", ""categor...","60660666,59657556,60759383,57315690,59260778"
3,macro,3.0,NaN,4580,0.102264,Law Enforcement,"Security authorities confront criminal, natura...",53.152813,-0.111563,0.509571,0.114349,0.145107,0.230973,domestic_security,subject::Kantonspolizei Zürich ¦ Bundesanwalts...,"{""subject"": [{""filler_type"": ""actor"", ""categor...","59273146,58741127,57988220,59997366,58248095"
4,macro,4.0,NaN,1142,0.025499,Electoral Rivalry,"Parties and candidates clash over mandates, of...",51.344397,-0.122931,0.681727,0.036950,0.124771,0.156552,political_power,subject::SVP (Swiss People's Party) ¦ SVP ¦ Sw...,"{""subject"": [{""filler_type"": ""actor"", ""categor...","60846857,60686635,59177600,58437597,58708241"


In [ ]:
# TABLES 2b/2c: slim extraction-derived tables (let the app skip the raw CSV)
slot_fillers = build_slot_fillers(extractions)
slot_fillers.to_parquet(cfg.SLOT_FILLERS, index=False)
print(f"slot_fillers: {len(slot_fillers):,} rows -> {cfg.SLOT_FILLERS}")

entity_ftype = build_entity_ftype(extractions)
entity_ftype.to_parquet(cfg.ENTITY_FTYPE, index=False)
print(f"entity_ftype: {len(entity_ftype):,} labels -> {cfg.ENTITY_FTYPE}")

In [6]:
# TABLE 3: cluster_publisher
cluster_publisher = build_cluster_publisher(article_table)
cluster_publisher.to_parquet(cfg.CLUSTER_PUBLISHER, index=False)
print(f"cluster_publisher: {len(cluster_publisher):,} rows -> {cfg.CLUSTER_PUBLISHER}")
cluster_publisher.head()

cluster_publisher: 3,414 rows -> data/viz/cluster_publisher.parquet


,level,cluster,publisher,n_articles,rel_by_publisher,rel_by_narrative,complexity,sentiment,anger,fear,joy,sadness
0,macro,0.0,CH Media,303,0.087979,0.074338,55.349879,-0.117653,0.571060,0.068363,0.177281,0.183296
1,macro,0.0,NZZ-Mediengruppe,664,0.108302,0.162905,56.449891,-0.167954,0.566959,0.066795,0.174139,0.192107
2,macro,0.0,Ringier,807,0.072435,0.197988,55.057683,-0.081091,0.534482,0.073108,0.202128,0.190283
3,macro,0.0,SRG,1317,0.139350,0.323111,54.481490,-0.067971,0.485401,0.087367,0.214058,0.213173
4,macro,0.0,TX Group,817,0.085568,0.200442,54.442345,-0.139141,0.574548,0.071225,0.172840,0.181387


In [7]:
# TABLE 4: cluster_week
cluster_week = build_cluster_week(article_table)
cluster_week.to_parquet(cfg.CLUSTER_WEEK, index=False)
print(f"cluster_week: {len(cluster_week):,} rows -> {cfg.CLUSTER_WEEK}")
cluster_week.head()

cluster_week: 18,017 rows -> data/viz/cluster_week.parquet


,level,cluster,iso_week,n_articles,within_week_share
0,macro,0.0,2025-06-02,59,0.086892
1,macro,0.0,2025-06-09,58,0.086826
2,macro,0.0,2025-06-16,88,0.104637
3,macro,0.0,2025-06-23,63,0.077874
4,macro,0.0,2025-06-30,49,0.061947


In [8]:
# SEARCH BUILD 1/3: load embedding model (same as cluster pipeline)
from sentence_transformers import SentenceTransformer
search_model = SentenceTransformer(cfg.EMBEDDING_MODEL)
print('Loaded', cfg.EMBEDDING_MODEL)

Loaded intfloat/multilingual-e5-small


In [ ]:
# SEARCH BUILD 2/3: entity index, vocab, and entity embeddings
from lib.precompute_lib import build_entity_index, build_entity_vocab, embed_entities

entity_index = build_entity_index(article_table, extractions)
entity_index.to_parquet(cfg.ENTITY_INDEX, index=False)
print(f'entity_index: {len(entity_index):,} rows -> {cfg.ENTITY_INDEX}')

entity_vocab = build_entity_vocab(entity_index)
entity_vocab.to_parquet(cfg.ENTITY_VOCAB, index=False)
print(f'entity_vocab: {len(entity_vocab):,} distinct entities -> {cfg.ENTITY_VOCAB}')

entity_emb = embed_entities(entity_vocab, search_model, cfg.EMBEDDING_BATCH)
np.save(cfg.ENTITY_EMBEDDINGS, entity_emb)
print(f'entity_embeddings: {entity_emb.shape} -> {cfg.ENTITY_EMBEDDINGS}')

# record which model built these embeddings (app checks this matches)
import json as _json
_json.dump({'model': cfg.EMBEDDING_MODEL, 'dim': int(entity_emb.shape[1])},
           open(cfg.SEARCH_META, 'w'))
print('search_meta:', cfg.EMBEDDING_MODEL)

# entity->article bridge (for entity search article scoping)
from lib.precompute_lib import build_entity_article_bridge
entity_bridge = build_entity_article_bridge(article_table, extractions)
entity_bridge.to_parquet(cfg.ENTITY_ARTICLE_BRIDGE, index=False)
print(f'entity_article_bridge: {len(entity_bridge):,} rows -> {cfg.ENTITY_ARTICLE_BRIDGE}')


entity_index: 711,657 rows -> data/viz/entity_index.parquet
entity_vocab: 265,706 distinct entities -> data/viz/entity_vocab.parquet


Batches:   0%|          | 0/4152 [00:00<?, ?it/s]

entity_embeddings: (265706, 384) -> data/viz/entity_embeddings.npy
search_meta: intfloat/multilingual-e5-small
entity_article_bridge: 416,174 rows -> data/viz/entity_article_bridge.parquet


In [10]:
# SEARCH BUILD 3/3: thematic cluster centroids (built with the e5-small search model)
# Narratives are embedded fresh with the SEARCH model + passage prefix so that
# thematic queries (embedded with the query prefix) land in the same space.
from lib.precompute_lib import embed_narratives_for_centroids, build_cluster_centroids

narr_emb_small, narr_ids_small = embed_narratives_for_centroids(
    article_table, extractions, search_model, cfg.EMBEDDING_BATCH)
np.save(cfg.NARRATIVE_EMB_SMALL, narr_emb_small)
np.save(cfg.NARRATIVE_EMB_IDS, narr_ids_small)
print(f'narrative embeddings (search model): {narr_emb_small.shape}')

centroids, centroid_keys = build_cluster_centroids(article_table, narr_emb_small, narr_ids_small)
np.save(cfg.CLUSTER_CENTROIDS, centroids)
centroid_keys.to_parquet(cfg.CLUSTER_CENTROID_KEYS, index=False)
print(f'cluster_centroids: {centroids.shape} -> {cfg.CLUSTER_CENTROIDS}')
print(f'centroid_keys: {len(centroid_keys):,} rows -> {cfg.CLUSTER_CENTROID_KEYS}')

Batches:   0%|          | 0/700 [00:00<?, ?it/s]

narrative embeddings (search model): (44786, 384)
cluster_centroids: (594, 384) -> data/viz/cluster_centroids.npy
centroid_keys: 594 rows -> data/viz/cluster_centroid_keys.parquet


In [11]:
# 2D SEMANTIC MAP: project e5-large narrative embeddings to 2D for the overview map
# (visualisation only — does not affect clustering)
from lib.precompute_lib import build_2d_map

map_xy, map_centroids = build_2d_map(
    article_table, embeddings, embedding_ids,
    n_neighbors=cfg.MAP_N_NEIGHBORS, min_dist=cfg.MAP_MIN_DIST)

# subsample the article cloud if very large (keeps the app responsive)
if len(map_xy) > cfg.MAP_CLOUD_SAMPLE:
    map_xy_out = map_xy.sample(cfg.MAP_CLOUD_SAMPLE, random_state=42)
else:
    map_xy_out = map_xy

map_xy_out.to_parquet(cfg.MAP_XY, index=False)
map_centroids.to_parquet(cfg.MAP_CENTROIDS, index=False)
print(f'map_xy: {len(map_xy_out):,} article points -> {cfg.MAP_XY}')
print(f'map_centroids: {len(map_centroids):,} cluster centroids -> {cfg.MAP_CENTROIDS}')

c:\Users\Sam\miniforge3\envs\cas-narrative-analysis\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


map_xy: 15,000 article points -> data/viz/map_xy.parquet
map_centroids: 594 cluster centroids -> data/viz/map_centroids.parquet


In [12]:
# sanity summary
print("Precompute complete. Four tables written to", cfg.VIZ_DIR)
for name, df in [("article_table", article_table), ("cluster_meta", cluster_meta),
                 ("cluster_publisher", cluster_publisher), ("cluster_week", cluster_week)]:
    print(f"  {name:20s} {len(df):>8,} rows  {df.shape[1]} cols")

Precompute complete. Four tables written to data/viz
  article_table          44,786 rows  20 cols
  cluster_meta              594 rows  17 cols
  cluster_publisher       3,414 rows  12 cols
  cluster_week           18,017 rows  5 cols
